In [1]:
import json
from uuid import uuid4
import httpx

## A2A(Agent-to-Agent) 기초

에이전트끼리 대화할 때 가장 중요한 것은 "상대방이 무엇을 할 수 있는지(Capability)"를 아는 것.

- Agent Card (명함): 에이전트의 이름, 버전, 제공하는 기술(Skills), 그리고 통신할 수 있는 주소(Endpoints)를 담은 JSON 파일입니다.
- Discovery (탐색): 클라이언트가 .well-known/agent-card.json 경로를 통해 에이전트의 정보를 읽어오는 과정입니다.


In [2]:
BASE_URL = "http://localhost:9999"
CARD_PATH = "/.well-known/agent-card.json"  # 서버가 이 경로로 카드 제공
DEFAULT_SEND_PATH = "/messages"             # 기본 REST 엔드포인트
DEFAULT_STREAM_PATH = "/messages/stream"    # 기본 스트리밍 엔드포인트(있으면)

In [3]:
# !pip install -qU fastapi a2a-sdk

In [4]:
import subprocess
!fuser -k 9999/tcp

import sys, subprocess
print("using python:", sys.executable)
subprocess.Popen([sys.executable, "hello_server.py"])

using python: /anaconda/envs/kt_env/bin/python


<Popen: returncode: None args: ['/anaconda/envs/kt_env/bin/python', 'hello_s...>

In [5]:
async with httpx.AsyncClient(base_url=BASE_URL, timeout=30) as http:
    # 1) 에이전트 카드 조회
    r = await http.get(CARD_PATH)
    r.raise_for_status()
    card = r.json()
    print("Agent Card:\n" + json.dumps(card, indent=2, ensure_ascii=False))
    # 에이전트 카드를 통해 어떤 스킬이 있고 어떤 엔드포인트에 요청을 해야하는지 알 수 있음
    
    # 카드로부터 엔드포인트 정보를 추출
    http_meta = card.get("http", {}) if isinstance(card, dict) else {}
    endpoints = http_meta.get("endpoints", {}) if isinstance(http_meta, dict) else {}
    send_path = endpoints.get("sendMessage", DEFAULT_SEND_PATH)
    
    # 표준 데이터 형식으로 메시지 전송
    payload = {
            "message": {
                "role": "user",
                "parts": [{"kind": "text", "text": "10달러가 한국 원으로 얼마인가요?"}],
                "messageId": uuid4().hex,
            }
        }
    
    # 엔드포인트에 요청
    resp = await http.post(send_path, json=payload)
    resp.raise_for_status()
    print("\n\n====== /messages response ======")
    print(json.dumps(resp.json(), indent=2, ensure_ascii=False))        

ConnectError: All connection attempts failed